# D. Emissions woIHS

**Overview.** Estimates emissions for the vessels that could **not** be matched to the IHS register, so the full activity-based method of notebook C cannot be applied. It first builds a simple statistical model from C's output: the **median** per-hour emission of each pollutant by AIS-derived vessel group (Cargo / Tanker / Passenger / Fishing / Tug / Others) and operational phase (At Berth / Anchored / Manoeuvring / Sea), with outliers excluded. It then assigns each unmatched vessel-hour the corresponding median, scaled by the time gap to the next position, to produce comparable per-hour emissions. Restricted to positions inside a country EEZ.

**Series.** Notebook **D** of the five-part series (A–E) estimating ship emissions in Pacific Island EEZs per the Fourth IMO GHG Study 2020:
- **A. AIS Data Prep** — hourly vessel-position panel tagged with EEZ/coast/port and IHS vessel IDs
- **B. IHS Specs** — per-vessel specs, correction factors, and emission factors
- **C. Emissions wIHS** — activity-based emissions for IHS-matched vessels
- **D. Emissions woIHS** — emissions for vessels without an IHS match, via the median-per-type model *(this notebook)*
- **E. Emissions Statistics** — monthly country / vessel-type / fishing aggregations and dashboard CSVs

**Inputs.**
- With-IHS emissions from C: `…/emissions/Pacific/emissions_data/with_ihs/` — used to fit the median model.
- Hourly AIS parquet from A: `…/emissions/Pacific/hourly_ais_v2/` — filtered to vessels with no `LRIMOShipNo` (no IHS match).
- Global Fishing Watch effort: `…/worldbank/GFW_effort_event/{year}.csv`.
- Platform S3 root `"s3a://" + os.environ["AWS_WORKING_DIRECTORY_PATH"] + "worldbank/"`.

**Output.**
- Model medians: `…/emissions_data/model/median_{year}_{vessel_type}.csv` (and a combined `{year}_median.csv`).
- Per-hour emissions parquet partitioned by `year`/`month`/`Country` at `…/emissions_data/no_ihs/`, with the same `…_xhour` pollutant columns as the with-IHS output so the two can be concatenated in E.

**Requirements.**
- Runs on the **UN Global Platform** — account required: https://datalab.officialstatistics.org
- PySpark cluster with the platform `ais` library; the median model is computed in pandas, then read back as a Spark CSV (with an explicit `model_schema`) and joined onto the AIS panel. Uses dynamic partition overwrite for monthly re-runs.
- Country sub-regions of Kiribati (Phoenix/Gilbert/Line groups) are collapsed into `Kiribati`.

# Initialize - Pacific

_Imports, Spark session, dynamic-overwrite config, and paths/constants._


In [1]:
import pandas as pd

pd.set_option('display.max_columns', None) #Show all columns in pandas df
pd.set_option('display.max_rows', 100) #Show all columns in pandas df
#silent future warnings
# pd.set_option('future.no_silent_downcasting', True)
pd.options.display.float_format = "{:,.2f}".format

from IPython.core.interactiveshell import InteractiveShell #allow multiple outputs in one jupyter cell
InteractiveShell.ast_node_interactivity = "all"

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()


In [3]:
from ais import functions as af
from pyspark.sql import functions as F
from pyspark.sql import Window as W
from pyspark.sql import types as t

In [4]:
import numpy as np
from dateutil.relativedelta import relativedelta

In [5]:
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
spark.conf.set("spark.sql.parquet.output.committer.class",
               "org.apache.parquet.hadoop.ParquetOutputCommitter")
spark.conf.set("spark.sql.sources.commitProtocolClass",
               "org.apache.spark.sql.execution.datasources.SQLHadoopMapReduceCommitProtocol")

/opt/spark/python/lib/pyspark.zip/pyspark/sql/context.py:113: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.


Closing down clientserver connection


In [6]:
import os
ihs_version = "20260202"
wb_path = "s3a://" + os.environ["AWS_WORKING_DIRECTORY_PATH"] + "worldbank/"
project_path = wb_path + "emissions/Pacific/"
out_path = project_path + "emissions_data/"
temp_path = wb_path + "temp/"
ais_base_path = project_path+"hourly_ais_v2/"
wihs_path = f"{out_path}with_ihs/"
ghg_list= ['_ch4_e', '_co_e','_n2o_e','_nmvoc_e', '_pm10_e', '_pm25_e', '_nox_e','_bc_e','_co2_f','_sox_f','_bc_f']
vessel_type_list = ["Cargo","Tanker","Passenger","Fishing","Tug","Others"]

# Build Model

_Section that derives the median-emissions lookup model from the with-IHS output._


## Save Data in Temp, only necessary

_Extracts the needed columns from the with-IHS emissions, assigns an AIS-based vessel group, and stages them in temp partitioned by vessel group._


In [7]:
year=2026
mode="overwrite"
( spark.read
 .option("basepath",wihs_path)
 .parquet(wihs_path + f"year={year}") 
 .withColumn("_vessel_group_ais",F.when(F.col("_vessel_class_no").isin([1,3,4,11,13]),"Cargo") \
             .when(F.col("_vessel_class_no").isin([2,5,6,7]), "Tanker") \
                                        .when(F.col("_vessel_class_no").isin([8,9,10,12]), "Passenger") \
                                        .when(F.col("_vessel_class_no").isin([15]),"Fishing") \
                                        .when(F.col("_vessel_class_no").isin([16]),"Tug") \
                                        .otherwise("Others"))
 .select("_op_phase","vessel_id","_vessel_group_ais","year",*ghg_list)
 .write
 .mode(mode)
 .partitionBy("year","_vessel_group_ais")
 .parquet(temp_path)
)

## Use pandas to get median per vessel type and op phase excluding outlier

_In pandas, computes the median per-hour emissions by operational phase for each vessel group and saves the model CSVs._


In [8]:
year=2026
for vessel_type in vessel_type_list:
        df = pd.read_parquet(f"{temp_path}year={year}/_vessel_group_ais={vessel_type}/")
        df_agg = df.groupby("_op_phase")[ghg_list].agg('median')
        df_agg.assign(year=year).assign(_vessel_group_ais=vessel_type).to_csv(f"{out_path}model/median_{year}_{vessel_type}.csv")

Found credentials from IAM Role: eksctl-sparky-mc-sparkface-nodegr-NodeInstanceRole-15Y58CLME2WUS


In [9]:
df_agg # as of Feb 2026 run 2026 model, Jan data only

,_ch4_e,_co_e,_n2o_e,_nmvoc_e,_pm10_e,_pm25_e,_nox_e,_bc_e,_co2_f,_sox_f,_bc_f
_op_phase,,,,,,,,,,,
Anchored,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
At Berth,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
Manoeuvring,3.08,62.71,3.83,178.39,66.62,61.29,"1,419.94",0.00,"29,340.13",47.14,54.08
Sea,18.90,555.45,57.86,"1,062.11",386.24,355.34,"20,618.87",0.00,"613,810.60",326.56,111.18


In [9]:
df_agg # as of Feb 2026 run 2025 model

,_ch4_e,_co_e,_n2o_e,_nmvoc_e,_pm10_e,_pm25_e,_nox_e,_bc_e,_co2_f,_sox_f,_bc_f
_op_phase,,,,,,,,,,,
Anchored,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
At Berth,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
Manoeuvring,4.65,82.73,4.77,269.36,76.34,70.24,"1,685.53",0.00,"30,663.65",47.00,58.13
Sea,19.13,557.82,55.94,"1,095.39",373.14,343.29,"20,030.57",0.00,"584,993.63",281.29,86.65


In [10]:
year=2026
df = pd.concat([pd.read_csv(f"{out_path}model/median_{year}_{vessel_type}.csv") for vessel_type in vessel_type_list])
df2 = pd.melt(df,
                  id_vars=["_vessel_group_ais", "_op_phase","year"], 
                  var_name="ghg",
                  value_name="Value").pivot_table(index=['_vessel_group_ais','_op_phase','ghg'], columns='year',values="Value",aggfunc="sum")
df2.to_csv(f"{out_path}model/{year}_median.csv")

# Calculate Emissions

_Defines `ais_prep` (load AIS for unmatched vessels, assign vessel group and operational phase) and `calc_emissions` (join the median model and scale by the time gap)._


In [11]:
w = W.partitionBy("vessel_id").orderBy("datetime")
w_following = W.partitionBy('vessel_id','Country').orderBy("datetime").rowsBetween(W.currentRow, W.unboundedFollowing) 

def ais_prep(paths_to_read, gfw=None):
    ais = ( spark
           .read
           .option("basepath",ais_base_path).parquet(*paths_to_read)
           .select("vessel_id","imo","mmsi","vessel_name","vessel_type","length","width","ISO3","Country","eez_flag","coast_flag","Port_name","port_flag",
                   "year","month", F.expr("to_timestamp(date  || ' ' || hour)").alias("dt_pos_utc"), 'longitude','latitude', F.col('sog_f').alias("sog"),
                   F.col('draught_f').alias("draught"),'LRIMOShipNo','ihs_matchtype')
           .filter(F.col("LRIMOShipNo").isNull())
           .withColumn("vessel_id",F.concat(F.lit("mmsi-"), F.col("mmsi").cast(StringType())))
           .withColumn("_vessel_group_ais", F.when(F.col("vessel_type")=="Cargo","Cargo") \
                       .when(F.col("vessel_type")=="Fishing","Fishing") \
                       .when(F.col("vessel_type")=="Passenger","Passenger") \
                       .when(F.col("vessel_type")=="Tanker","Tanker") \
                       .when(F.col("vessel_type").isin(["Towing","Tug"]),"Tug") \
                       .otherwise("Others")
                      )
           .withColumn("_op_phase", F.when(F.col("sog").isNull(), None) \
                 .when((F.col("sog") <=1) & F.col("port_flag"), "At Berth") \
                 .when(F.col("sog") <=3, "Anchored") \
                 .when(F.col("sog").between(3,5) & (F.col("port_flag") | F.col("coast_flag")),"Manoeuvring") \
                 .otherwise("Sea")
                      )
           .na.fill(0,['imo'])
           .withColumnRenamed("Country","Country_sub")
           .withColumn("Country",F.when(F.col("Country_sub").isin(['Phoenix Group','Gilbert Islands','Line Group']),"Kiribati").otherwise(F.col("Country_sub")))
           .withColumn("date",F.to_date("dt_pos_utc"))
           .withColumnRenamed("dt_pos_utc","datetime")
           .withColumns({"eez_flag_next":F.lag("eez_flag",-1).over(w),
                         "datetime_next":F.lag("datetime",-1).over(w)
                        })
           .withColumn("draught_nozero", F.col("draught")).replace(0, None, subset=['draught_nozero'])
           .withColumn("draught_bfill", F.first(F.col("draught_nozero"), True).over(w_following))
          )
    if gfw:
        ais_gfw = ais.join(gfw, on=['date','mmsi','Country_sub'], how="left").withColumn("_w_fishing", F.col("fishing_hours").isNotNull())
    else:
        ais_gfw = ais.withColumn("fishing_hours", F.lit(None).cast(DoubleType())).withColumn("_w_fishing", F.lit(False))
        
    return ais_gfw

In [12]:
def calc_emissions(ais, model):
    ais_emissions = ais_gfw2.join(model, on=["year","_vessel_group_ais","_op_phase"], how="left") \
    .withColumns({
        '_hour_diff': (F.col('datetime_next').cast(t.LongType()) -F.col('datetime').cast(t.LongType())) / 3600,
        '_time_multiplier':F.when(F.col("_hour_diff") > 24,24) \
                            .when(F.col("_hour_diff") > 1,F.round(F.col("_hour_diff"),0)) \
                            .otherwise(1),
    }| {
        f"{ghg}_xhour":F.col(f"{ghg}") * F.col("_time_multiplier") for ghg in ghg_list}
    )
    
    return ais_emissions

In [13]:
def read_gfw(year):
    gfw = spark.read.option("header",True).csv(f"{wb_path}GFW_effort_event/{year}.csv") \
    .select(F.col("mmsi").cast(IntegerType()),
            F.col("Country").alias("Country_sub"),
            F.col("hours").alias("fishing_hours").cast(DoubleType()),
            F.to_date("date").alias("date"))
    
    return gfw

In [14]:
model_schema = StructType([
    StructField('_op_phase', StringType(), True), 
    StructField('_ch4_e', DoubleType(), True), 
    StructField('_co_e', DoubleType(), True), 
    StructField('_n2o_e', DoubleType(), True), 
    StructField('_nmvoc_e', DoubleType(), True), 
    StructField('_pm10_e', DoubleType(), True), 
    StructField('_pm25_e', DoubleType(), True), 
    StructField('_nox_e', DoubleType(), True), 
    StructField('_bc_e', DoubleType(), True), 
    StructField('_co2_f', DoubleType(), True), 
    StructField('_sox_f', DoubleType(), True), 
    StructField('_bc_f', DoubleType(), True), 
    StructField('year', IntegerType(), True), 
    StructField('_vessel_group_ais', StringType(), True)])

In [15]:
date_list = pd.date_range("2025-11-01","2026-01-31", freq="M")
date_list

DatetimeIndex(['2025-11-30', '2025-12-31', '2026-01-31'], dtype='datetime64[ns]', freq='M')

## first

_Initial run for the seed month (mode=overwrite)._


In [16]:
year=2025
gfw = read_gfw(year)
model = spark.read.option("header",True).schema(model_schema).csv(f"{out_path}model/median_{year}_*.csv")

In [17]:
date = date_list[0]
next_date = date + relativedelta(months=1)
paths_to_read = [ais_base_path+f"year={date.year}/month={date.month}", ais_base_path+f"year={next_date.year}/month={next_date.month}"]
ais_gfw= ais_prep(paths_to_read, gfw)
ais_gfw2 = ais_gfw.filter((F.col("year")==date.year) & (F.col("month")==date.month))
ais_emissions = calc_emissions(ais_gfw2, model)

In [18]:
ais_emissions.select(["year","_vessel_group_ais","_op_phase"] + ghg_list).show()

+----+-----------------+---------+------------------+-----------------+-----------------+------------------+------------------+-----------------+-----------------+-----+-----------------+------------------+-----------------+
|year|_vessel_group_ais|_op_phase|            _ch4_e|            _co_e|           _n2o_e|          _nmvoc_e|           _pm10_e|          _pm25_e|           _nox_e|_bc_e|           _co2_f|            _sox_f|            _bc_f|
+----+-----------------+---------+------------------+-----------------+-----------------+------------------+------------------+-----------------+-----------------+-----+-----------------+------------------+-----------------+
|2025|           Others| Anchored|               0.0|              0.0|              0.0|               0.0|               0.0|              0.0|              0.0|  0.0|              0.0|               0.0|              0.0|
|2025|           Others| Anchored|               0.0|              0.0|              0.0|           

In [20]:
mode="overwrite"
date = date_list[0]
next_date = date + relativedelta(months=1)
paths_to_read = [ais_base_path+f"year={date.year}/month={date.month}", ais_base_path+f"year={next_date.year}/month={next_date.month}"]
ais_gfw= ais_prep(paths_to_read, gfw)
ais_gfw2 = ais_gfw.filter((F.col("year")==date.year) & (F.col("month")==date.month))
ais_emissions = calc_emissions(ais_gfw2, model)
ais_emissions.repartition(1).write.mode(mode).partitionBy("year","month","Country").parquet(f"{out_path}no_ihs")

In [27]:
for date in date_list[1:-1]:
    print(date)

2025-02-28 00:00:00
2025-03-31 00:00:00
2025-04-30 00:00:00
2025-05-31 00:00:00
2025-06-30 00:00:00
2025-07-31 00:00:00
2025-08-31 00:00:00
2025-09-30 00:00:00
2025-10-31 00:00:00


## loop

_Loops over months applying the model and appending to `no_ihs/`._


In [22]:
year = 2025
gfw = read_gfw(year)
model = spark.read.option("header",True).schema(model_schema).csv(f"{out_path}model/median_{year}_*.csv")

In [23]:
mode="overwrite"
for date in date_list[:-1]:
    print(date)
    next_date = date + relativedelta(months=1)
    paths_to_read = [ais_base_path+f"year={date.year}/month={date.month}", ais_base_path+f"year={next_date.year}/month={next_date.month}"]
    ais_gfw= ais_prep(paths_to_read, gfw)
    ais_gfw2 = ais_gfw.filter((F.col("year")==date.year) & (F.col("month")==date.month))
    ais_emissions = calc_emissions(ais_gfw2, model)
    ais_emissions.repartition(1).write.mode(mode).partitionBy("year","month","Country").parquet(f"{out_path}no_ihs")

2025-11-30 00:00:00
2025-12-31 00:00:00


## last

_Runs the final month and checks the CO₂ total._


In [16]:
date = date_list[-1]
date

Timestamp('2026-01-31 00:00:00', freq='M')

In [17]:
gfw = read_gfw(2026)

In [18]:
year=2026
mode="overwrite"

model = spark.read.option("header",True).schema(model_schema).csv(f"{out_path}model/median_{year}_*.csv")
paths_to_read = [ais_base_path+f"year={date.year}/month={date.month}"]
ais_gfw= ais_prep(paths_to_read, gfw)
ais_gfw2 = ais_gfw.filter((F.col("year")==date.year) & (F.col("month")==date.month))

ais_emissions = calc_emissions(ais_gfw2, model)
ais_emissions.repartition(1).write.mode(mode).partitionBy("year","month","Country").parquet(f"{out_path}no_ihs")

In [19]:
sdf = spark.read.parquet(f"{out_path}no_ihs/year=2026")

In [21]:
df = sdf.groupBy("Country").agg(F.sum("_co2_f_xhour")).toPandas()

In [27]:
df[~df['Country'].isnull()]['sum(_co2_f_xhour)'].sum() / 10**6

87620.42682166872